# Module 1 — Notebook 2: Indexing & Boolean Retrieval

**Covers:** Theory sections 3, 4, 5, 6 · Week 2 Lab (Part 2) · Week 3 Lab (Boolean + TF-IDF + BM25)

**Prerequisite:** Run `01_preprocessing.ipynb` first — this notebook loads the processed documents it saved.

| Section | Topic | Manning |
|---------|-------|--------|
| 1 | Term-Document Matrix | Ch. 1 |
| 2 | Inverted Index (from scratch) | Ch. 1–2 |
| 3 | Sort-Based Indexing (BSBI) | Ch. 4 |
| 4 | Boolean Retrieval (AND / OR / NOT) | Ch. 1 |
| 5 | Query Optimisation & Skip Pointers | Ch. 2 |
| 6 | Positional Index & Phrase Queries | Ch. 2 |
| 7 | Index Compression (Gap + VByte) | Ch. 5 |
| 8 | Bridge: TF-IDF & BM25 preview | Ch. 6–7 |

In [2]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import json, re, math
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Path helper — works regardless of kernel working directory ────────────────
def _find_file(*parts):
    """Walk upward from cwd until the relative path resolves to an existing file."""
    p = Path.cwd()
    for _ in range(8):
        c = p.joinpath(*parts)
        if c.exists():
            return c
        p = p.parent
    raise FileNotFoundError(f"Could not find {'/'.join(parts)} — cwd was {Path.cwd()}")

def short_id(doc_id: str, n: int = 14) -> str:
    """Shorten a long doc_id for display."""
    return doc_id[:n] + '…' if len(doc_id) > n else doc_id

# ── Load the mini-corpus handoff from 01_preprocessing.ipynb ─────────────────
_docs_path = Path('/tmp/mini_corpus_processed.json')
if _docs_path.exists():
    PROCESSED_DOCS = json.loads(_docs_path.read_text(encoding='utf-8'))
    print(f"Loaded {len(PROCESSED_DOCS)} documents from {_docs_path.name}")
    print(f"  path: {_docs_path}")
else:
    print("WARNING: /tmp/mini_corpus_processed.json not found.")
    print("Run 01_preprocessing.ipynb first to generate it, then re-run this cell.")
    PROCESSED_DOCS = None

# Quick inspect
if PROCESSED_DOCS:
    print(f"\n{'ID (short)':<16} {'schema':<15} {'caption':<32} n_tokens")
    print("-" * 72)
    for d in PROCESSED_DOCS[:12]:
        print(f"{short_id(d['doc_id']):<16} {d['schema']:<15} {d['caption'][:31]:<32} {len(d['tokens'])}")
    if len(PROCESSED_DOCS) > 12:
        print(f"  … ({len(PROCESSED_DOCS) - 12} more documents)")

Loaded 100 documents from documents.jsonl
  path: /Users/alireza/Library/CloudStorage/GoogleDrive-ali.esterabi@gmail.com/My Drive/QMUL_tem_2/IR_project//tmp/mini_corpus_processed.json

ID (short)       schema          caption                          n_tokens
------------------------------------------------------------------------
NK-223CQDBzp8M…  Company         Myanmar Yatai International Hol  63
NK-223yQP6hRaM…  Company         Товариство з обмеженою відповід  25
NK-224TRezPqwz…  Person          SANAVBARI NIKITENKO              4
NK-226GXBdQ5p6…  Company         Open Joint Stock Company “Elekt  380
NK-228ZdYZVXaZ…  Company         Приватне підприємство "Магістар  20
NK-228jBYSTdUS…  Company         Акціонерне товариство "Електроа  25
NK-229j9NEWBex…  LegalEntity     MICHAEL DAVID MUMMERT            9
NK-22BiLDQqi5m…  Person          ARNITA LEFF                      6
NK-22C9zkXEo48…  Vessel          DONG CHANG                       3
NK-22HtK7WrxZ2…  Person          Michael Kuajien 

In [6]:
# ── FALLBACK: regenerate PROCESSED_DOCS from sample_10k.json ─────────────────
# Runs only if /tmp/mini_corpus_processed.json was not found above.
# Uses a simplified pipeline (no spaCy lemmatisation) so spaCy is not required.
# For lemmatised tokens identical to the full pipeline, run 01_preprocessing.ipynb first.

if PROCESSED_DOCS is None:
    import unicodedata
    import nltk
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    STOP_WORDS = set(stopwords.words('english'))

    def _normalize_fb(text):
        text = unicodedata.normalize('NFC', text).lower()
        text = unicodedata.normalize('NFD', text)
        text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
        text = re.sub(r'[^\w\s]', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    def _tokenize_fb(text):
        return [t for t in _normalize_fb(text).split() if t not in STOP_WORDS and len(t) > 1]

    try:
        _raw_path = _find_file("data", "raw_data", "sample_10k.json")
        _raw_docs  = [json.loads(line) for line in _raw_path.open(encoding="utf-8")][:100]
        PROCESSED_DOCS = []
        for e in _raw_docs:
            props  = e.get('properties', {})
            tokens = []
            for field in ['name', 'alias', 'previousName']:
                for v in props.get(field, []):
                    if isinstance(v, str) and v.strip():
                        tokens.extend(_normalize_fb(v).split())
            for field in ['notes', 'description', 'summary']:
                for v in props.get(field, []):
                    if isinstance(v, str) and v.strip():
                        tokens.extend(_tokenize_fb(v))
            for v in props.get('topics', []):
                if isinstance(v, str):
                    tokens.append(_normalize_fb(v))
            PROCESSED_DOCS.append({
                'doc_id':    e.get('id', ''),
                'caption':   e.get('caption', ''),
                'schema':    e.get('schema', ''),
                'text_blob': ' '.join(tokens),
                'tokens':    tokens,
                'identifiers': {},
                'metadata':  {'country': props.get('country', []),
                              'topics':  props.get('topics', [])},
            })
        print(f"Rebuilt {len(PROCESSED_DOCS)} documents from sample_10k.json (first 100 rows, simplified fallback).")
        print("Tip: run 01_preprocessing.ipynb for lemmatised tokens and then re-run this notebook.")
    except FileNotFoundError:
        raise RuntimeError(
            "Neither /tmp/mini_corpus_processed.json nor data/raw_data/sample_10k.json found. "
            "Check your data/ folder."
        )

---
## Section 1: Term-Document Matrix

The simplest representation: a matrix where rows = documents, columns = terms,  
values = 1 if the term appears in the document, 0 otherwise.  
(Manning §1.1, Week 2 Lab Part 2)

**Problem:** For 1.2M docs × 500K terms → 600 billion cells → impractical.  
**Solution:** Use the sparsity — most cells are 0. Store only the 1s → inverted index.

In [7]:
# Build vocabulary from all processed docs
vocabulary = sorted(set(t for doc in PROCESSED_DOCS for t in doc['tokens']))
doc_ids    = [doc['doc_id'] for doc in PROCESSED_DOCS]

print(f'Vocabulary size: {len(vocabulary)} terms')
print(f'Documents:       {len(PROCESSED_DOCS)}')
print(f'Matrix cells:    {len(vocabulary) * len(PROCESSED_DOCS):,} ({len(vocabulary)}×{len(PROCESSED_DOCS)})')

# Binary term-document matrix
binary_matrix = [
    [1 if term in set(doc['tokens']) else 0 for term in vocabulary]
    for doc in PROCESSED_DOCS
]

df_matrix = pd.DataFrame(binary_matrix, index=doc_ids, columns=vocabulary)
print(f'\nNon-zero entries: {df_matrix.values.sum():,} ({100*df_matrix.values.sum()/(df_matrix.shape[0]*df_matrix.shape[1]):.1f}% density)')

# Show a slice of the matrix (first 20 terms)
print(f'\nTerm-document matrix (first 20 terms):')
df_matrix[vocabulary[:20]]

Vocabulary size: 1004 terms
Documents:       100
Matrix cells:    100,400 (1004×100)

Non-zero entries: 1,392 (1.4% density)

Term-document matrix (first 20 terms):


,1110,21 professional medical group,4,76 durable medical equipment supplier 60 medicare certified home health agency home health agency 60 medicare certified home health agency home health agency waivered services organization waivered service organization,abd,abu,access,action,acupuncturist,add,addition,adresse,affiliate,after,ahmed,aircraft,ais,aktsionerne,aktsionernoe,al
NK-223CQDBzp8MRkdJMDiqXn3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
NK-223yQP6hRaMuiALDCJ6xbY,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
NK-224TRezPqwzhQZ37exWxtX,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
NK-226GXBdQ5p6NjgrTpTQNVW,0,0,0,0,0,0,0,1,0,0,1,1,0,1,0,1,0,1,0,0
NK-228ZdYZVXaZBSBgVwapnks,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NK-24gRTsgNqZhU3FB4iULKBT,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
NK-24k9PZGKdYTmzHPDJfTUSR,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
NK-24kqBFxEsFzjqBYhaNHkC9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
NK-24mFPrTQyTsaqNvHMFstSy,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [8]:
# Term occurrence dictionary: term → list of doc_ids containing it
# This is the raw material for the inverted index
term_occurrence = defaultdict(list)
for doc in PROCESSED_DOCS:
    for term in set(doc['tokens']):   # set() to avoid duplicate doc_ids
        term_occurrence[term].append(doc['doc_id'])

# Document frequency: how many docs contain each term?
doc_frequency = {term: len(docs) for term, docs in term_occurrence.items()}

print('Term occurrence (postings lists) — top 15 by document frequency:')
print(f"{'Term':<20} {'DF':>4}  Postings (doc IDs truncated)")
print("-" * 72)
for term, df in sorted(doc_frequency.items(), key=lambda x: -x[1])[:15]:
    preview = [short_id(d) for d in term_occurrence[term][:6]]
    suffix  = ', …' if df > 6 else ''
    print(f"{term:<20} {df:>4}  {preview}{suffix}")

Term occurrence (postings lists) — top 15 by document frequency:
Term                   DF  Postings (doc IDs truncated)
------------------------------------------------------------------------
debarment              52  ['NK-223CQDBzp8M…', 'NK-22BiLDQqi5m…', 'NK-22HtK7WrxZ2…', 'NK-22JRrAHhpxz…', 'NK-22MHXvQQufB…', 'NK-22SrWxmRnUf…'], …
sanction               46  ['NK-223CQDBzp8M…', 'NK-223yQP6hRaM…', 'NK-226GXBdQ5p6…', 'NK-228ZdYZVXaZ…', 'NK-228jBYSTdUS…', 'NK-229j9NEWBex…'], …
company                13  ['NK-223yQP6hRaM…', 'NK-226GXBdQ5p6…', 'NK-228jBYSTdUS…', 'NK-22MHXvQQufB…', 'NK-22cQD3a8GDR…', 'NK-22xbFMnMDYA…'], …
общество               13  ['NK-223yQP6hRaM…', 'NK-226GXBdQ5p6…', 'NK-228jBYSTdUS…', 'NK-22MHXvQQufB…', 'NK-22WmG9GWt6y…', 'NK-22YnEcmYTBX…'], …
export control         11  ['NK-229j9NEWBex…', 'NK-22MHXvQQufB…', 'NK-22T6dTsc6um…', 'NK-22cQD3a8GDR…', 'NK-23RAhcNT2DH…', 'NK-23ZhBn7gtgV…'], …
tovarystvo             10  ['NK-223yQP6hRaM…', 'NK-226GXBdQ5p6…', 'NK-228jBYSTdUS

In [9]:
# Lab Exercise (Week 2 Lab Q3): Term Frequency
# Calculate TF for the most common term in the corpus
target_term = sorted(doc_frequency.keys(), key=lambda t: -doc_frequency[t])[0]
# Pick first document that actually contains that term
_target = next((d for d in PROCESSED_DOCS if target_term in d['tokens']), PROCESSED_DOCS[0])
doc_tokens = _target['tokens']

tf = doc_tokens.count(target_term) / len(doc_tokens) if doc_tokens else 0
print(f"TF('{target_term}', '{_target['caption'][:35]}') = "
      f"{doc_tokens.count(target_term)} / {len(doc_tokens)} = {tf:.4f}")

# Lab Exercise (Week 2 Lab Q4): Document Frequency
df_top = doc_frequency.get(target_term, 0)
print(f"DF('{target_term}') = {df_top} (appears in {df_top} of {len(PROCESSED_DOCS)} documents)")

TF('debarment', 'Myanmar Yatai International Holding') = 1 / 63 = 0.0159
DF('debarment') = 52 (appears in 52 of 100 documents)


---
## Section 2: Building an Inverted Index from Scratch

An **inverted index** maps each term to its **postings list** — the sorted list of doc_ids containing it.  
(Manning §1.2)

```
term        → [doc_id₁, doc_id₂, ...]  sorted
────────────────────────────────────────
sanction    → [E001, E002, E003, E004, E005, E006, E007, E008]
russian     → [E001, E005, E008]
vessel      → [E003, E007]
tanker      → [E003]
```

**Key property:** Postings lists are **sorted** → enables fast set intersection (AND queries).

In [10]:
# Build a proper inverted index with term frequency per document
# Structure: {term: {doc_id: count, ...}}

def build_inverted_index(docs: list) -> dict:
    """
    Build an inverted index from processed documents.
    Returns: {term: {'df': int, 'postings': [(doc_id, tf), ...]}}
    """
    raw = defaultdict(lambda: defaultdict(int))   # term → doc_id → count

    for doc in docs:
        for term in doc['tokens']:
            raw[term][doc['doc_id']] += 1

    index = {}
    for term, doc_counts in raw.items():
        postings = sorted(doc_counts.items())   # sort by doc_id
        index[term] = {
            'df':       len(postings),
            'postings': postings,               # [(doc_id, tf), ...]
        }
    return index

INDEX = build_inverted_index(PROCESSED_DOCS)

# Inspect top 6 terms from the actual index
inspect_terms = sorted(INDEX.keys(), key=lambda t: -INDEX[t]['df'])[:6]
print(f"{'Term':<22} {'DF':>4}  Postings (short_id, tf) …")
print("-" * 72)
for term in inspect_terms:
    entry = INDEX[term]
    preview = [(short_id(d), tf) for d, tf in entry['postings'][:5]]
    suffix  = ', …' if entry['df'] > 5 else ''
    print(f"{term:<22} {entry['df']:>4}  {preview}{suffix}")

Term                     DF  Postings (short_id, tf) …
------------------------------------------------------------------------
debarment                52  [('NK-223CQDBzp8M…', 1), ('NK-22BiLDQqi5m…', 1), ('NK-22HtK7WrxZ2…', 1), ('NK-22JRrAHhpxz…', 1), ('NK-22MHXvQQufB…', 1)], …
sanction                 46  [('NK-223CQDBzp8M…', 1), ('NK-223yQP6hRaM…', 1), ('NK-226GXBdQ5p6…', 1), ('NK-228ZdYZVXaZ…', 1), ('NK-228jBYSTdUS…', 1)], …
общество                 13  [('NK-223yQP6hRaM…', 1), ('NK-226GXBdQ5p6…', 6), ('NK-228jBYSTdUS…', 3), ('NK-22MHXvQQufB…', 2), ('NK-22WmG9GWt6y…', 1)], …
company                  13  [('NK-223yQP6hRaM…', 1), ('NK-226GXBdQ5p6…', 3), ('NK-228jBYSTdUS…', 1), ('NK-22MHXvQQufB…', 3), ('NK-22cQD3a8GDR…', 2)], …
export control           11  [('NK-229j9NEWBex…', 1), ('NK-22MHXvQQufB…', 1), ('NK-22T6dTsc6um…', 1), ('NK-22cQD3a8GDR…', 1), ('NK-23RAhcNT2DH…', 1)], …
товариство               10  [('NK-223yQP6hRaM…', 1), ('NK-226GXBdQ5p6…', 1), ('NK-228jBYSTdUS…', 1), ('NK-

In [11]:
# Index statistics
total_postings = sum(e['df'] for e in INDEX.values())
print(f"Index statistics:")
print(f"  Vocabulary size : {len(INDEX):,} terms")
print(f"  Total postings  : {total_postings:,}")
print(f"  Avg postings/term: {total_postings/len(INDEX):.1f}")

# Distribution of df values
df_dist = Counter(e['df'] for e in INDEX.values())
print(f"\nDocument frequency distribution:")
for df_val in sorted(df_dist):
    print(f"  DF={df_val}: {df_dist[df_val]} terms")

Index statistics:
  Vocabulary size : 1,004 terms
  Total postings  : 1,392
  Avg postings/term: 1.4

Document frequency distribution:
  DF=1: 878 terms
  DF=2: 71 terms
  DF=3: 22 terms
  DF=4: 9 terms
  DF=5: 2 terms
  DF=6: 8 terms
  DF=7: 1 terms
  DF=8: 4 terms
  DF=9: 2 terms
  DF=10: 2 terms
  DF=11: 1 terms
  DF=13: 2 terms
  DF=46: 1 terms
  DF=52: 1 terms


---
## Section 3: Sort-Based Indexing (BSBI)

For large corpora that don't fit in RAM, we use **Blocked Sort-Based Indexing** (Manning §4.2):

1. Parse documents in blocks, emit (term_id, doc_id) pairs  
2. Sort each block and write to disk  
3. Merge sorted blocks → final index  

This is conceptually what happens when building a large IR system index.

In [12]:
# Simulate BSBI on the toy corpus
BLOCK_SIZE = 3   # process 3 documents per block (tiny for illustration)

# Step 1: Emit (term, doc_id) pairs in blocks
blocks = []
current_block = []

for doc in PROCESSED_DOCS:
    for term in set(doc['tokens']):
        current_block.append((term, doc['doc_id']))
    if len([d for d in PROCESSED_DOCS[:PROCESSED_DOCS.index(doc)+1]]) % BLOCK_SIZE == 0:
        blocks.append(sorted(current_block))   # sort block
        current_block = []

if current_block:
    blocks.append(sorted(current_block))       # final partial block

print(f"BSBI: {len(PROCESSED_DOCS)} docs split into {len(blocks)} blocks of ≤{BLOCK_SIZE} docs each\n")
for i, block in enumerate(blocks):
    print(f"Block {i+1} ({len(block)} pairs, first 5):")
    for pair in block[:5]:
        print(f"  {pair}")
    print()

# Step 2: Merge blocks (like merge sort) → final sorted stream
import heapq
merged = list(heapq.merge(*blocks))   # sorted merge

# Step 3: Group into postings lists
bsbi_index = defaultdict(list)
for term, doc_id in merged:
    if not bsbi_index[term] or bsbi_index[term][-1] != doc_id:
        bsbi_index[term].append(doc_id)

print(f"BSBI index: {len(bsbi_index)} terms (matches hand-built index: {len(bsbi_index) == len(INDEX)})")

BSBI: 100 docs split into 34 blocks of ≤3 docs each

Block 1 (47 pairs, first 5):
  ('city', 'NK-223CQDBzp8MRkdJMDiqXn3')
  ('co', 'NK-223CQDBzp8MRkdJMDiqXn3')
  ('company', 'NK-223yQP6hRaMuiALDCJ6xbY')
  ('construction of buildings', 'NK-223CQDBzp8MRkdJMDiqXn3')
  ('crime', 'NK-224TRezPqwzhQZ37exWxtX')

Block 2 (180 pairs, first 5):
  ('action', 'NK-226GXBdQ5p6NjgrTpTQNVW')
  ('addition', 'NK-226GXBdQ5p6NjgrTpTQNVW')
  ('adresse', 'NK-226GXBdQ5p6NjgrTpTQNVW')
  ('after', 'NK-226GXBdQ5p6NjgrTpTQNVW')
  ('aircraft', 'NK-226GXBdQ5p6NjgrTpTQNVW')

Block 3 (16 pairs, first 5):
  ('add', 'NK-229j9NEWBexje4PuNpeHxg')
  ('addition', 'NK-229j9NEWBexje4PuNpeHxg')
  ('arnita', 'NK-22BiLDQqi5mhKfDeYuqs23')
  ('chang', 'NK-22C9zkXEo48ioJPccyPR4c')
  ('cowan', 'NK-22BiLDQqi5mhKfDeYuqs23')

Block 4 (22 pairs, first 5):
  ('clothing', 'NK-22JT8frx32YRjjYnnwxsyy')
  ('co', 'NK-22JT8frx32YRjjYnnwxsyy')
  ('cotton textile apparel', 'NK-22JT8frx32YRjjYnnwxsyy')
  ('debarment', 'NK-22HtK7WrxZ2sU3rmhz6PuZ'

---
## Section 4: Boolean Retrieval

**Boolean model:** Query = Boolean expression over terms (AND, OR, NOT).  
(Manning §1.3, Week 3 Lab)

Implemented via **set operations on postings lists**:
- `AND` → intersection (postings list merge)
- `OR`  → union
- `NOT` → complement relative to all docs

**Key property:** AND uses sorted postings → O(p₁ + p₂) merge, not O(p₁ × p₂) intersection.

In [13]:
# Helper: get postings as a set of doc_ids
ALL_DOC_IDS = {doc['doc_id'] for doc in PROCESSED_DOCS}

def get_postings(term: str) -> set:
    """Return set of doc_ids containing term."""
    if term not in INDEX:
        return set()
    return {doc_id for doc_id, _ in INDEX[term]['postings']}

def boolean_AND(*terms) -> set:
    """Intersection of all postings lists — documents containing ALL terms."""
    if not terms:
        return ALL_DOC_IDS
    result = get_postings(terms[0])
    for t in terms[1:]:
        result &= get_postings(t)
    return result

def boolean_OR(*terms) -> set:
    """Union — documents containing ANY of the terms."""
    result = set()
    for t in terms:
        result |= get_postings(t)
    return result

def boolean_NOT(term: str) -> set:
    """Complement — documents NOT containing the term."""
    return ALL_DOC_IDS - get_postings(term)

def boolean_AND_NOT(include_term: str, exclude_term: str) -> set:
    """Documents containing include_term but NOT exclude_term."""
    return get_postings(include_term) - get_postings(exclude_term)

def show_results(query_label: str, result: set):
    captions = {doc['doc_id']: doc['caption'] for doc in PROCESSED_DOCS}
    print(f"  Query: {query_label}")
    if result:
        for doc_id in sorted(result):
            print(f"    → {doc_id}: {captions[doc_id]}")
    else:
        print("    (no results)")
    print()

In [14]:
# Boolean queries on the toy corpus
# Pick the two most frequent terms from the real index for the AND/NOT examples
_by_df   = sorted(INDEX.keys(), key=lambda t: -INDEX[t]['df'])
_t1      = _by_df[0]    # most frequent (almost always 'sanction' in OpenSanctions)
_t2      = _by_df[1]    # second most frequent
_t3      = _by_df[2] if len(_by_df) > 2 else _t2

print("=== Boolean Retrieval Results ===\n")
print(f"(top index terms used in examples: '{_t1}' DF={INDEX[_t1]['df']}, "
      f"'{_t2}' DF={INDEX[_t2]['df']}, '{_t3}' DF={INDEX[_t3]['df']})\n")

# AND: both terms must appear
show_results(f"{_t1} AND {_t2}",
             boolean_AND(_t1, _t2))

# OR: either term
show_results("vessel OR ship",
             boolean_OR('vessel', 'ship'))

# AND NOT: top-term entities that are NOT vessels
show_results(f"{_t1} AND NOT vessel",
             boolean_AND_NOT(_t1, 'vessel'))

# 3-way AND
show_results(f"{_t1} AND {_t2} AND {_t3}",
             boolean_AND(_t1, _t2, _t3))

# NOT (all docs without the top term — usually empty for OpenSanctions)
show_results(f"NOT {_t1}",
             boolean_NOT(_t1))

=== Boolean Retrieval Results ===

(top index terms used in examples: 'debarment' DF=52, 'sanction' DF=46, 'общество' DF=13)

  Query: debarment AND sanction
    → NK-223CQDBzp8MRkdJMDiqXn3: Myanmar Yatai International Holding Group Co., LTD.
    → NK-22HtK7WrxZ2sU3rmhz6PuZ: Michael Kuajien
    → NK-22MHXvQQufBgTjUWgUbWb8: Limited Liability Company Specialized Developer Alabuga South Park
    → NK-22T6dTsc6umYgEA33vME3t: Digital Marketing Awards FZ LLC
    → NK-22cQD3a8GDRSuoW5mus3F3: Ekvik Limited Liability Company
    → NK-2369Vca9iPnQoaUJDAHTaB: Star Dragon Corporation Limited
    → NK-23p2d4vMT5sJtQ845GyzJt: Scientific and Production Association of Measuring Equipment JSC
    → NK-23rgYEXa9AHtupZKgS8Tbc: BAHRAMI ALI SHAYESTEH
    → NK-23wEenKCkJ2dbXMy4qdiE7: Golden Luxury Jewellery Trading L.L.C
    → NK-24D3iX76oPbjQLMjmK2ijd: USAMAH 'ABD-AL-WAHID AL-JAZA'IRI BELKACEM
    → NK-24PVSqqr5CmqxAyN4Tzan6: Nasser Al Sheikh Ali
    → NK-24UMVkuxF3svsAtg55i3Zi: Albahr Alaahmar Energy FZE


In [15]:
# Lab Exercise (Week 3 Lab): Verify AND implementation by tracing manually
# Uses the top-2 terms from the real index (_t1 and _t2 defined in previous cell)

print(f"'{_t1}' postings: ", [short_id(d) for d in sorted(get_postings(_t1))])
print(f"'{_t2}' postings: ", [short_id(d) for d in sorted(get_postings(_t2))])

manual_and = get_postings(_t1) & get_postings(_t2)
print(f"Manual AND ('{_t1}' ∩ '{_t2}'): ", [short_id(d) for d in sorted(manual_and)])

func_and = boolean_AND(_t1, _t2)
print(f"Via boolean_AND:               ", [short_id(d) for d in sorted(func_and)])
print("Match:", manual_and == func_and)

'debarment' postings:  ['NK-223CQDBzp8M…', 'NK-22BiLDQqi5m…', 'NK-22HtK7WrxZ2…', 'NK-22JRrAHhpxz…', 'NK-22MHXvQQufB…', 'NK-22SrWxmRnUf…', 'NK-22T6dTsc6um…', 'NK-22Tion5LJoM…', 'NK-22cQD3a8GDR…', 'NK-22j9bR6whUf…', 'NK-22nRrnXxfTi…', 'NK-22sjj2aodJm…', 'NK-22wTQUzAxxg…', 'NK-22xTZjBMmFg…', 'NK-233T8qPscJE…', 'NK-2369Vca9iPn…', 'NK-236GpFjsG9V…', 'NK-23975toCAxG…', 'NK-23Ab6pm659o…', 'NK-23BSgxe29kH…', 'NK-23DxmPckzT3…', 'NK-23Gj7ZDNkpw…', 'NK-23KkjXHEZQd…', 'NK-23NRktrKQgB…', 'NK-23QCKo3YFMJ…', 'NK-23WZSwPtpLf…', 'NK-23YhzeoZWza…', 'NK-23b3kRvzCbH…', 'NK-23b47Q7KTge…', 'NK-23ezoxpLv8H…', 'NK-23h7ifEwCt3…', 'NK-23hJQLey537…', 'NK-23i5vJArY9n…', 'NK-23jff7RiRmZ…', 'NK-23p2d4vMT5s…', 'NK-23rgYEXa9AH…', 'NK-23wEenKCkJ2…', 'NK-247pdTAo4kT…', 'NK-24BMpXYsemA…', 'NK-24D3iX76oPb…', 'NK-24GSFhigDX7…', 'NK-24GV3qws4cq…', 'NK-24HBT9wzJBL…', 'NK-24PVSqqr5Cm…', 'NK-24Q3XY8bwTx…', 'NK-24UMVkuxF3s…', 'NK-24ZomtS59Xs…', 'NK-24aPBAKYZKf…', 'NK-24cEBmXGjPP…', 'NK-24cf2zYe7LE…', 'NK-24k9PZGKdYT…', 'NK-24k

### 4.1 Query Optimisation & Skip Pointers

**Query optimisation** (Manning §2.3): When evaluating an AND query over multiple terms,  
process terms in **increasing DF order** — smallest postings list first.

Example: `russian AND sanction AND financing`
- `russian`: DF=3
- `financing`: DF=2  
- `sanction`: DF=8

Optimal order: `financing` → `russian` → `sanction` (prune early)

**Skip pointers** (Manning §2.3.2): Augment postings lists with periodic "skip" links  
to jump over sections that cannot satisfy the intersection.

In [16]:
# Query optimisation: evaluate AND in increasing DF order
def optimised_AND(*terms) -> set:
    """
    Boolean AND with query optimisation:
    sort terms by DF ascending, intersect smallest first.
    """
    # Sort by document frequency (ascending)
    sorted_terms = sorted(terms, key=lambda t: INDEX[t]['df'] if t in INDEX else 0)
    print(f"  Optimised order (DF ascending): {[(t, INDEX[t]['df'] if t in INDEX else 0) for t in sorted_terms]}")
    return boolean_AND(*sorted_terms)

# Use the top-3 terms from the real index to demonstrate optimised ordering
print("=== Optimised AND Query ===")
result   = optimised_AND(_t1, _t2, _t3)
captions = {doc['doc_id']: doc['caption'] for doc in PROCESSED_DOCS}
print(f"  Results: {[captions[d][:30] for d in sorted(result)]}")

=== Optimised AND Query ===
  Optimised order (DF ascending): [('общество', 13), ('sanction', 46), ('debarment', 52)]
  Results: ['Limited Liability Company Spec', 'Ekvik Limited Liability Compan', 'Scientific and Production Asso']


---
## Section 5: Positional Index & Phrase Queries

A standard inverted index cannot answer **phrase queries** like `"crude oil"`  
because it doesn't know which words are adjacent.  
(Manning §2.4)

**Solution:** Store **positions** of each term in each document:

```
crude  → {E003: [6], E007: [3]}
oil    → {E003: [7], E007: [4]}
```

For `"crude oil"`: need docs where position(crude) + 1 = position(oil).

In [17]:
# Build positional index: {term: {doc_id: [position, ...]}}
def build_positional_index(docs: list) -> dict:
    pos_index = defaultdict(lambda: defaultdict(list))
    for doc in docs:
        for pos, term in enumerate(doc['tokens']):
            pos_index[term][doc['doc_id']].append(pos)
    return {term: dict(postings) for term, postings in pos_index.items()}

POS_INDEX = build_positional_index(PROCESSED_DOCS)

# Show positional postings for query terms
print("Positional postings for selected terms:\n")
for term in ['crude', 'oil', 'russian', 'sanction', 'vessel']:
    if term in POS_INDEX:
        print(f"  '{term}':")
        for doc_id, positions in POS_INDEX[term].items():
            print(f"    {doc_id}: {positions}")

Positional postings for selected terms:

  'crude':
    NK-24cf2zYe7LEKhr2zFck73j: [43, 218]
  'oil':
    NK-24cf2zYe7LEKhr2zFck73j: [44, 76, 81, 102, 107, 124, 144, 151, 163, 177, 190, 191, 219, 257]
  'russian':
    NK-226GXBdQ5p6NjgrTpTQNVW: [294, 305, 307, 321, 332, 334, 346, 355, 365]
    NK-22c6upVtxbVMhbZb3Z6dn6: [21]
    NK-24cf2zYe7LEKhr2zFck73j: [75, 80, 82, 101, 158]
  'sanction':
    NK-223CQDBzp8MRkdJMDiqXn3: [60]
    NK-223yQP6hRaMuiALDCJ6xbY: [23]
    NK-226GXBdQ5p6NjgrTpTQNVW: [377]
    NK-228ZdYZVXaZBSBgVwapnks: [19]
    NK-228jBYSTdUSvbZvsKsiHh6: [23]
    NK-229j9NEWBexje4PuNpeHxg: [7]
    NK-22HtK7WrxZ2sU3rmhz6PuZ: [24]
    NK-22KNpaKJL84d8wpFudvLjN: [8]
    NK-22MHXvQQufBgTjUWgUbWb8: [84]
    NK-22T6dTsc6umYgEA33vME3t: [37]
    NK-22YnEcmYTBXeCBBeZU5wCN: [35]
    NK-22c6upVtxbVMhbZb3Z6dn6: [24]
    NK-22cQD3a8GDRSuoW5mus3F3: [13]
    NK-22tXKtJr4xMCbfKZr5RzEz: [2]
    NK-22xbFMnMDYA4FMDBQ5keE3: [39]
    NK-235NUsjt6Rg6aKTiS5ZF4f: [14]
    NK-235hApQ4jp8oMsJmToKGdJ: 

In [18]:
# Phrase query: find documents containing terms at consecutive positions
def phrase_query(phrase: str, k: int = 1) -> list:
    """
    Find documents containing the phrase (terms within k positions of each other).
    k=1: exact adjacency (phrase query)
    k>1: proximity query
    """
    terms = phrase.lower().split()
    if not terms:
        return []

    # Start with candidate docs containing all terms
    candidate_docs = set(POS_INDEX.get(terms[0], {}).keys())
    for t in terms[1:]:
        candidate_docs &= set(POS_INDEX.get(t, {}).keys())

    results = []
    for doc_id in candidate_docs:
        # Check if terms appear in the right order within k positions
        positions = [POS_INDEX[t][doc_id] for t in terms]
        # Try all starting positions of the first term
        for start_pos in positions[0]:
            match = True
            prev  = start_pos
            for next_positions in positions[1:]:
                # Next term must appear within k positions after prev
                valid = [p for p in next_positions if prev < p <= prev + k]
                if not valid:
                    match = False
                    break
                prev = min(valid)
            if match:
                results.append(doc_id)
                break
    return sorted(results)

captions = {doc['doc_id']: doc['caption'] for doc in PROCESSED_DOCS}
print("=== Phrase Queries ===\n")
for phrase in ['crude oil', 'russian sanction', 'arms smuggling', 'money launder']:
    result = phrase_query(phrase)
    docs_str = ', '.join(f"{d}:{captions[d]}" for d in result) if result else '(no match)'
    print(f"  \"{phrase}\"  →  {docs_str}")

print()
print("=== Proximity Queries (k=3) ===\n")
for phrase in ['sanction energy', 'vessel oil']:
    result = phrase_query(phrase, k=3)
    docs_str = ', '.join(f"{d}:{captions[d]}" for d in result) if result else '(no match)'
    print(f"  \"{phrase}\" within 3 positions  →  {docs_str}")

=== Phrase Queries ===

  "crude oil"  →  NK-24cf2zYe7LEKhr2zFck73j:Nurkez
  "russian sanction"  →  (no match)
  "arms smuggling"  →  (no match)
  "money launder"  →  (no match)

=== Proximity Queries (k=3) ===

  "sanction energy" within 3 positions  →  (no match)
  "vessel oil" within 3 positions  →  NK-24cf2zYe7LEKhr2zFck73j:Nurkez


---
## Section 6: Index Compression

Storing raw doc IDs in postings lists wastes space.  
Two key compression techniques: (Manning §5.3)

### Gap Encoding (Delta Encoding)
Instead of storing absolute doc IDs, store the **gap** between consecutive IDs.

```
Raw:  [8,   15,  23,  27,  50]
Gaps: [8,    7,   8,   4,  23]   (first gap = first ID)
```

Gaps are smaller numbers → need fewer bits to encode.

### Variable-Byte (VByte) Encoding
Use 7 bits per byte for data, 1 continuation bit.  
Small numbers → 1 byte. Large numbers → 2+ bytes.

In [19]:
# Map actual doc_ids → sequential integers for the numeric compression demo
doc_id_map = {doc['doc_id']: i + 1 for i, doc in enumerate(PROCESSED_DOCS)}

# Convert a sample postings list to numeric
sample_term = 'sanction'
raw_ids = sorted(doc_id_map[d] for d, _ in INDEX[sample_term]['postings'] if d in doc_id_map)
print(f"Term: '{sample_term}'")
print(f"Raw doc IDs : {raw_ids}")

# Gap encoding
def gap_encode(ids: list) -> list:
    if not ids:
        return []
    gaps = [ids[0]]
    for i in range(1, len(ids)):
        gaps.append(ids[i] - ids[i-1])
    return gaps

def gap_decode(gaps: list) -> list:
    if not gaps:
        return []
    ids = [gaps[0]]
    for g in gaps[1:]:
        ids.append(ids[-1] + g)
    return ids

gaps = gap_encode(raw_ids)
print(f"Gap encoded : {gaps}")
print(f"Decoded back: {gap_decode(gaps)}")
print(f"Roundtrip OK: {gap_decode(gap_encode(raw_ids)) == raw_ids}")

Term: 'sanction'
Raw doc IDs : [1, 2, 4, 5, 6, 7, 10, 13, 14, 16, 19, 21, 22, 29, 33, 35, 36, 37, 38, 44, 46, 52, 55, 57, 58, 61, 65, 67, 69, 70, 71, 72, 73, 77, 78, 83, 85, 89, 91, 92, 94, 95, 97, 98, 99, 100]
Gap encoded : [1, 1, 2, 1, 1, 1, 3, 3, 1, 2, 3, 2, 1, 7, 4, 2, 1, 1, 1, 6, 2, 6, 3, 2, 1, 3, 4, 2, 2, 1, 1, 1, 1, 4, 1, 5, 2, 4, 2, 1, 2, 1, 2, 1, 1, 1]
Decoded back: [1, 2, 4, 5, 6, 7, 10, 13, 14, 16, 19, 21, 22, 29, 33, 35, 36, 37, 38, 44, 46, 52, 55, 57, 58, 61, 65, 67, 69, 70, 71, 72, 73, 77, 78, 83, 85, 89, 91, 92, 94, 95, 97, 98, 99, 100]
Roundtrip OK: True


In [20]:
# Variable-Byte (VByte) encoding
def vbyte_encode(n: int) -> bytes:
    """Encode a non-negative integer using variable-byte encoding."""
    result = []
    while True:
        result.insert(0, n % 128)   # 7 bits
        n //= 128
        if n == 0:
            break
    result[-1] |= 128               # set continuation bit on last byte
    return bytes(result)

def vbyte_decode(data: bytes) -> list:
    """Decode a sequence of VByte-encoded integers."""
    numbers, n = [], 0
    for byte in data:
        if byte < 128:
            n = 128 * n + byte
        else:
            n = 128 * n + (byte - 128)
            numbers.append(n)
            n = 0
    return numbers

print("VByte encoding examples:")
print(f"{'Number':>10}  {'Bits (fixed 32)':>16}  {'VByte bytes':>12}  {'Bits (VByte)':>13}  {'Saving':>8}")
print("-" * 68)
for num in [1, 7, 127, 128, 255, 1000, 16384]:
    vb     = vbyte_encode(num)
    saving = (4 - len(vb)) / 4 * 100
    print(f"{num:>10}  {32:>16}  {len(vb):>12}  {8*len(vb):>13}  {saving:>7.0f}%")

# Encode the gap-encoded postings list
encoded = b''.join(vbyte_encode(g) for g in gaps)
decoded = vbyte_decode(encoded)
print(f"\nGap list    : {gaps}")
print(f"VByte bytes : {list(encoded)} ({len(encoded)} bytes)")
print(f"Decoded gaps: {decoded}")
print(f"Final IDs   : {gap_decode(decoded)}")

VByte encoding examples:
    Number   Bits (fixed 32)   VByte bytes   Bits (VByte)    Saving
--------------------------------------------------------------------
         1                32             1              8       75%
         7                32             1              8       75%
       127                32             1              8       75%
       128                32             2             16       50%
       255                32             2             16       50%
      1000                32             2             16       50%
     16384                32             3             24       25%

Gap list    : [1, 1, 2, 1, 1, 1, 3, 3, 1, 2, 3, 2, 1, 7, 4, 2, 1, 1, 1, 6, 2, 6, 3, 2, 1, 3, 4, 2, 2, 1, 1, 1, 1, 4, 1, 5, 2, 4, 2, 1, 2, 1, 2, 1, 1, 1]
VByte bytes : [129, 129, 130, 129, 129, 129, 131, 131, 129, 130, 131, 130, 129, 135, 132, 130, 129, 129, 129, 134, 130, 134, 131, 130, 129, 131, 132, 130, 130, 129, 129, 129, 129, 132, 129, 133, 130, 132, 130

In [21]:
# Compression comparison across all terms in the index
results = []
for term, entry in INDEX.items():
    ids  = sorted(doc_id_map[d] for d, _ in entry['postings'] if d in doc_id_map)
    if not ids:
        continue
    raw_bytes  = len(ids) * 4                          # 32-bit integers
    gap_bytes  = sum(len(vbyte_encode(g)) for g in gap_encode(ids))
    results.append({'term': term, 'df': len(ids),
                    'raw_bytes': raw_bytes, 'compressed_bytes': gap_bytes,
                    'ratio': gap_bytes / raw_bytes})

df_comp = pd.DataFrame(results).sort_values('df', ascending=False)
avg_ratio = df_comp['ratio'].mean()
print(f"Average compression ratio: {avg_ratio:.2f} ({(1-avg_ratio)*100:.0f}% size reduction)")
print(f"\nTop terms by DF with compression stats:")
print(df_comp.head(10).to_string(index=False))

Average compression ratio: 0.25 (75% size reduction)

Top terms by DF with compression stats:
          term  df  raw_bytes  compressed_bytes  ratio
     debarment  52        208                52   0.25
      sanction  46        184                46   0.25
       company  13         52                13   0.25
      общество  13         52                13   0.25
export control  11         44                11   0.25
    tovarystvo  10         40                10   0.25
    товариство  10         40                10   0.25
       limited   9         36                 9   0.25
           ltd   9         36                 9   0.25
  ограниченнои   8         32                 8   0.25


---
## Section 7: Bridge — TF-IDF & BM25 Preview

Boolean retrieval returns **sets** — no ranking.  
To rank results, we need a **scoring function**.  
(Manning Ch. 6–7, Week 3 Lab)

### TF-IDF

$$\text{TF-IDF}(t, d) = \underbrace{\frac{tf_{t,d}}{\sum_k tf_{k,d}}}_{\text{TF: term freq in doc}} \times \underbrace{\log\frac{N}{df_t}}_{\text{IDF: rarity}}$$

- Terms appearing often in a doc AND rarely across the corpus get high weight
- Stop words (high DF) get low weight → automatic weighting without explicit stop word removal

### BM25 (Best Match 25)

$$\text{BM25}(t, d) = \text{IDF}(t) \times \frac{tf_{t,d} \cdot (k_1 + 1)}{tf_{t,d} + k_1 \cdot (1 - b + b \cdot \frac{|d|}{\text{avgdl}})}$$

- $k_1 \approx 1.2$: term frequency saturation (diminishing returns)
- $b \approx 0.75$: document length normalisation
- State-of-the-art for sparse retrieval

In [22]:
# TF-IDF scoring
N = len(PROCESSED_DOCS)

def tf(term: str, doc: dict) -> float:
    count = doc['tokens'].count(term)
    return count / len(doc['tokens']) if doc['tokens'] else 0.0

def idf(term: str) -> float:
    df_t = INDEX[term]['df'] if term in INDEX else 0
    return math.log(N / (df_t + 1)) + 1   # +1 smoothing

def tfidf_score(query_terms: list, doc: dict) -> float:
    return sum(tf(t, doc) * idf(t) for t in query_terms if t in INDEX)

# BM25 scoring
k1   = 1.2
b    = 0.75
avgdl = sum(len(doc['tokens']) for doc in PROCESSED_DOCS) / N

def bm25_score(query_terms: list, doc: dict) -> float:
    score = 0.0
    dl = len(doc['tokens'])
    for t in query_terms:
        if t not in INDEX:
            continue
        tf_t = doc['tokens'].count(t)
        df_t = INDEX[t]['df']
        idf_t = math.log((N - df_t + 0.5) / (df_t + 0.5) + 1)
        bm25_tf = tf_t * (k1 + 1) / (tf_t + k1 * (1 - b + b * dl / avgdl))
        score  += idf_t * bm25_tf
    return score

# Compare Boolean vs TF-IDF vs BM25 on the same query
# Use the top-2 index terms so the query is guaranteed to return results
query = [_t1, _t2]
print(f"Query: {query}\n")
print(f"{'ID (short)':<16} {'caption':<34} {'TF-IDF':>8} {'BM25':>8}")
print("-" * 72)

scored = [(doc, tfidf_score(query, doc), bm25_score(query, doc)) for doc in PROCESSED_DOCS]
scored.sort(key=lambda x: x[2], reverse=True)

for doc, tfidf, bm25 in scored:
    if tfidf > 0 or bm25 > 0:
        print(f"{short_id(doc['doc_id']):<16} {doc['caption'][:33]:<34} {tfidf:>8.4f} {bm25:>8.4f}")

Query: ['debarment', 'sanction']

ID (short)       caption                              TF-IDF     BM25
------------------------------------------------------------------------
NK-2369Vca9iPn…  Star Dragon Corporation Limited      0.4843   2.0363
NK-24ZomtS59Xs…  Sun Science International Co., Li    0.3082   1.8671
NK-24k9PZGKdYT…  Angel Daniel Balestrini Jaramillo    0.2421   1.7576
NK-24UMVkuxF3s…  Albahr Alaahmar Energy FZE           0.2119   1.6915
NK-22cQD3a8GDR…  Ekvik Limited Liability Company      0.1994   1.6602
NK-24cEBmXGjPP…  LOLA LOLITA 1110, S. DE R.L. DE C    0.1614   1.5460
NK-24PVSqqr5Cm…  Nasser Al Sheikh Ali                 0.1356   1.4465
NK-22HtK7WrxZ2…  Michael Kuajien                      0.1304   1.4236
NK-23wEenKCkJ2…  Golden Luxury Jewellery Trading L    0.1130   1.3388
NK-22tXKtJr4xM…  Khalid SBAA                          0.5850   1.2146
NK-235iqUbQS4S…  Zviad TSEKVAVA                       0.5850   1.2146
NK-23rgYEXa9AH…  BAHRAMI ALI SHAYESTEH               

In [23]:
# Lab Exercise (Week 3 Lab Q1-Q4): BM25 parameter sensitivity
# Q: What is the purpose of k1 and b in BM25?

# `query` is inherited from the previous cell (top-2 index terms)
target_doc = PROCESSED_DOCS[0]  # first document in the toy corpus

print(f"BM25 parameter sensitivity  query={query}  doc='{target_doc['caption'][:40]}'\n")
print(f"{'k1':>6} {'b':>6} {'score':>10}  interpretation")
print("-" * 55)
for k1_val in [0.5, 1.2, 2.0]:
    for b_val in [0.0, 0.75, 1.0]:
        dl   = len(target_doc['tokens'])
        s    = 0.0
        for t in query:
            if t not in INDEX: continue
            tf_t  = target_doc['tokens'].count(t)
            df_t  = INDEX[t]['df']
            idf_t = math.log((N - df_t + 0.5) / (df_t + 0.5) + 1)
            bm25tf = tf_t * (k1_val + 1) / (tf_t + k1_val * (1 - b_val + b_val * dl / avgdl))
            s     += idf_t * bm25tf
        print(f"{k1_val:>6.1f} {b_val:>6.2f} {s:>10.4f}")

print("\nConclusion:")
print("  k1: controls TF saturation. Higher k1 = TF keeps increasing (less saturation).")
print("  b:  controls doc length normalisation. b=0 = no normalisation, b=1 = full.")

BM25 parameter sensitivity  query=['debarment', 'sanction']  doc='Myanmar Yatai International Holding Grou'

    k1      b      score  interpretation
-------------------------------------------------------
   0.5   0.00     1.4300
   0.5   0.75     1.0496
   0.5   1.00     0.9641
   1.2   0.00     1.4300
   1.2   0.75     0.8977
   1.2   1.00     0.7986
   2.0   0.00     1.4300
   2.0   0.75     0.8291
   2.0   1.00     0.7272

Conclusion:
  k1: controls TF saturation. Higher k1 = TF keeps increasing (less saturation).
  b:  controls doc length normalisation. b=0 = no normalisation, b=1 = full.


---
## Summary

| Section | What we built | Key insight |
|---------|--------------|-------------|
| Term-doc matrix | Binary representation | Dense → impractical at scale |
| Inverted index | term → [(doc_id, tf)] | Sparse → efficient |
| BSBI | Sort-merge indexing | Handles out-of-RAM collections |
| Boolean retrieval | AND/OR/NOT via set ops | Exact, no ranking |
| Query optimisation | Sort by DF ascending | Prune early, big speedup |
| Positional index | Stores term positions | Enables phrase queries |
| Compression | Gap + VByte | 30-50% space reduction |
| TF-IDF / BM25 | Ranked retrieval | BM25 = industry standard baseline |

**Phase 5 of the project** builds a full BM25 index over 1.24M real OpenSanctions entities  
using `rank-bm25` — the same algorithm you've just implemented from scratch.